# Passo 5 – Hiperparâmetros da árvore (Fase A: só acerto)

Cenário do Passo 4 (candle × janela × re-treino escolhidos pelo R²), 20 variáveis, alvo retorno %. **Nenhum custo, nenhum dinheiro.**

| Hiperparâmetro | O que controla | Grade |
|---|---|---|
| `max_depth` | quantas perguntas em sequência cada árvore faz | 3, 4, 6, 8, 12 |
| `min_samples_leaf` | fração mínima de candles em cada folha | 0,1%, 0,5%, 1%, 2%, 5% |
| `max_features` | fração das variáveis sorteadas em cada divisão | 0,33, 0,5, 1,0 |
| `n_estimators` | nº de árvores | 300 (fixo: mais árvores só estabilizam a média) |

**Regra de escolha (definida antes de rodar):** entre as combinações com **R² > 0 em pelo menos 3 dos 4 trimestres**, a de maior **R² médio na vizinhança** (a própria e as vizinhas de ±1 profundidade e ±1 folha, mesmo `max_features`). Uma região boa é mais confiável que um pico isolado.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, time, itertools
from sklearn.ensemble import RandomForestRegressor
import rf_zero as rz
g4 = pd.read_csv("saidas/A4_grade.csv", index_col=[0, 1, 2])
cand = g4[g4["trimestres R²>0"] >= 3]
M, J, F = (cand if len(cand) else g4)["R² %"].idxmax()
FREQ = {"mensal": "MS", "trimestral": "QS", "semestral": "6MS"}[F]
print(f"Cenário do Passo 4: {M} min × janela {J} m × re-treino {F}")
win_1m = rz.carregar("../../dados/WINN_M1/WIN$N_M1.csv")
base, cols = rz.montar_base(win_1m, M)
PROF = [3, 4, 6, 8, 12]; FOLHA = [0.001, 0.005, 0.01, 0.02, 0.05]; MF = [0.33, 0.5, 1.0]
ARQ = "saidas/A5_previsoes.pkl"
prevs = pd.read_pickle(ARQ) if os.path.exists(ARQ) else {}
t0 = time.time()
for p, f, mf in itertools.product(PROF, FOLHA, MF):
    if (p, f, mf) in prevs: continue
    fab = lambda n, p=p, f=f, mf=mf: RandomForestRegressor(n_estimators=300, max_depth=p, min_samples_leaf=f,
                                                         max_features=mf, n_jobs=-1, random_state=42)
    prevs[(p, f, mf)] = rz.walk_forward(base, cols, rz.VALIDACAO, J, FREQ, fabrica=fab)[["prev", "ret_prox", "mov_prox", "close"]]
    pd.to_pickle(prevs, ARQ)
print(f"{len(prevs)} combinações, {time.time() - t0:.0f} s")

In [ ]:
grade = pd.DataFrame({k: rz.resumo_acerto(v) for k, v in prevs.items()}).T
grade.index.names = ["prof", "folha", "max_feat"]
grade = grade.astype(float).sort_index()
viz = {}
for (p, f, mf) in grade.index:
    i, j = PROF.index(p), FOLHA.index(f)
    vz = [(PROF[a], FOLHA[b], mf) for a in range(max(i - 1, 0), min(i + 2, 5)) for b in range(max(j - 1, 0), min(j + 2, 5))]
    viz[(p, f, mf)] = grade.loc[vz, "R² %"].mean()
grade["R² vizinhança %"] = pd.Series(viz)
grade.to_csv("saidas/A5_grade.csv")
cand = grade[grade["trimestres R²>0"] >= 3]
ESCOLHA = (cand if len(cand) else grade)["R² vizinhança %"].idxmax()
print("Candidatas:", len(cand), "| Escolha pela regra →", ESCOLHA)
grade.sort_values("R² vizinhança %", ascending=False).head(10).round(3)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
for a, mf in zip(axs, MF):
    g = grade.xs(mf, level="max_feat")["R² %"].unstack("folha")
    a.imshow(g.values, cmap="RdYlGn", vmin=-0.8, vmax=0.8, aspect="auto")
    for i in range(g.shape[0]):
        for k in range(g.shape[1]):
            a.text(k, i, f"{g.values[i, k]:+.2f}", ha="center", va="center", fontsize=8)
    a.set_xticks(range(5)); a.set_xticklabels([f"{x*100:g}%" for x in FOLHA]); a.set_yticks(range(5)); a.set_yticklabels(PROF)
    a.set_xlabel("folha mínima"); a.set_ylabel("profundidade"); a.set_title(f"R² % · max_features {mf}")
plt.tight_layout(); plt.savefig("passo05_hiperparametros.png", dpi=120); plt.show()

In [ ]:
pd.concat({d: grade.groupby(level=d)[["R² %", "acerto direção %"]].mean() for d in ["prof", "folha", "max_feat"]}).round(3)

In [ ]:
import json
json.dump({"candle": int(M), "janela": int(J), "retreino": F, "max_depth": int(ESCOLHA[0]),
           "min_samples_leaf": float(ESCOLHA[1]), "max_features": float(ESCOLHA[2]), "n_lags": rz.N_LAGS},
          open("saidas/modelo_congelado.json", "w"), indent=1)
print(open("saidas/modelo_congelado.json").read())

## Conclusões do Passo 5

*(preenchidas após rodar)*